# Tutorial 1: Binary Softmax Convergence Radius

This notebook demonstrates the core result: the convergence radius
of softmax cross-entropy loss is $\rho = \pi / \Delta$, where $\Delta$
is the logit gap.

For binary softmax, the exact radius is
$\rho^* = \sqrt{\delta^2 + \pi^2} / \Delta_a$, and the lower bound
$\pi / \Delta_a$ is tight when $\delta \to 0$ (uniform predictions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from ghosts.radii import compute_rho_from_delta, compute_rho_out

PI = np.pi

## 1. Radius as a function of logit gap

The lower bound $\rho = \pi / \Delta$ decreases as the logit gap grows.
This means sharper predictions have a smaller safe step-size region.

In [ ]:
deltas = np.linspace(0.1, 10, 200)
rhoLower = PI / deltas
rhoExact = np.sqrt(deltas**2 + PI**2) / deltas  # binary exact

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(deltas, rhoLower, label=r"Lower bound $\pi/\Delta$", lw=2)
ax.plot(deltas, rhoExact, "--", label=r"Exact (binary) $\sqrt{\delta^2+\pi^2}/\Delta_a$", lw=2)
ax.set_xlabel(r"Logit gap $\Delta$")
ax.set_ylabel(r"Convergence radius $\rho$")
ax.set_title("Convergence radius shrinks as predictions sharpen")
ax.legend()
ax.set_ylim(0, 5)
plt.tight_layout()
plt.show()

## 2. Using the library

`compute_rho_out` computes $\rho$ directly from logits.

In [ ]:
# Uniform logits -> large rho (safe)
uniform = torch.tensor([[0.0, 0.0]])
print(f"Uniform: rho = {compute_rho_out(uniform, gap='maxmin', reduce='mean'):.2f}")

# Confident logits -> small rho (fragile)
confident = torch.tensor([[0.0, 10.0]])
print(f"Confident: rho = {compute_rho_out(confident, gap='maxmin', reduce='mean'):.4f}")

# Batch of samples
batch = torch.tensor([[0.0, 1.0], [0.0, 5.0], [0.0, 10.0]])
perSample = compute_rho_out(batch, gap='maxmin', reduce='per_sample')
print(f"Per-sample rho: {perSample}")

## 3. Loss landscape near a singularity

The cross-entropy loss along a direction $p$ has a complex singularity
at distance $\rho$ from the origin. Beyond this radius, the Taylor
series diverges.

In [ ]:
def ceLoss(z, target=0):
    """Binary cross-entropy as function of logit difference."""
    return np.log(1 + np.exp(-z)) if target == 0 else np.log(1 + np.exp(z))

z0 = 2.0  # starting logit gap
delta = z0  # max-min gap = z0 for binary [z0/2, -z0/2]
rho = PI / delta

taus = np.linspace(-3*rho, 3*rho, 500)
losses = [ceLoss(z0 + t) for t in taus]

# Taylor approximation (quadratic)
L0 = ceLoss(z0)
sig = 1 / (1 + np.exp(-z0))
dL = -(1 - sig)
d2L = sig * (1 - sig)
taylor2 = L0 + dL * taus + 0.5 * d2L * taus**2

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(taus, losses, 'k-', lw=2, label='True loss')
ax.plot(taus, taylor2, 'r--', lw=1.5, label='Quadratic Taylor')
ax.axvline(-rho, color='blue', ls=':', alpha=0.7, label=fr'$\pm\rho = \pm{rho:.2f}$')
ax.axvline(rho, color='blue', ls=':', alpha=0.7)
ax.set_xlabel(r'Step size $\tau$')
ax.set_ylabel('Loss')
ax.set_title(f'Loss landscape along a direction (logit gap = {z0})')
ax.legend()
ax.set_ylim(-0.1, 3)
plt.tight_layout()
plt.show()